# Monthly means of z, temp, sal

Maps of the **mean particle property** (depth, temperature, salinity) per spatial bin, grouped by release month.
Same weighted-`xhist` approach as the particle density: weight the histogram by the variable, divide by the count → mean value per bin.

In [2]:
import xarray as xr
from pathlib import Path
from xhistogram.xarray import histogram as xhist
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cf
from cartopy.mpl.ticker import LongitudeFormatter, LatitudeFormatter
import cmocean as cm

In [ ]:
from dask.distributed import Client
client = Client(n_workers=4, threads_per_worker=3, memory_limit=15e9)
client

In [ ]:
stores1 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/tracks_45678").glob("Parcels_run_*_*.zarr"))
stores2 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/tracks_67891").glob("Parcels_run_*_*.zarr"))
stores3 = sorted(Path("/work/bk1450/b383184/Amazon/Mercator/data/tracks_78876").glob("Parcels_run_*_*.zarr"))

stores = stores1 + stores2 + stores3
len(stores)

In [ ]:
ds_list = [xr.open_zarr(s) for s in stores]
ds = xr.concat(ds_list, dim="trajectory")
# NOTE: we KEEP z, temp, sal this time (do not drop them)
ds

In [ ]:
# trim trailing obs steps where no particle has data
num_valid_obs_steps = int(ds.lat.notnull().any("trajectory").sum().compute().data[()])
ds = ds.isel(obs=slice(None, num_valid_obs_steps))
ds

In [ ]:
# release time + month, exactly as in the density notebook
ds = ds.assign(start_time=ds.time.isel(obs=0).compute())
ds = ds.assign(start_month=ds.start_time.dt.month)

In [ ]:
lat_min = ds.lat.min().compute().data[()]
lat_max = ds.lat.max().compute().data[()]
lon_min = ds.lon.min().compute().data[()]
lon_max = ds.lon.max().compute().data[()]

lat_bins = np.linspace(lat_min, lat_max, 30)
lon_bins = np.linspace(lon_min, lon_max, 80)

## Weighted histogram → monthly mean per variable

In [ ]:
variables = ["z", "temp", "sal"]

def calc_var(d):
    """For each variable: binned sum and binned count, weighted by the variable.
    Positions where the variable is NaN are excluded by masking the coordinates."""
    out = {}
    for v in variables:
        w = d[v]
        lon = d.lon.where(w.notnull())   # drop samples with no value for this var
        lat = d.lat.where(w.notnull())
        count = xhist(lon, lat, bins=[lon_bins, lat_bins],
                      dim=["trajectory"], bin_dim_suffix="")
        wsum  = xhist(lon, lat, bins=[lon_bins, lat_bins],
                      dim=["trajectory"], weights=w, bin_dim_suffix="")
        out[f"{v}_sum"]   = wsum
        out[f"{v}_count"] = count
    return xr.Dataset(out)

In [ ]:
monthly = ds.groupby("start_month").apply(calc_var)

# collapse the obs axis (skip the initial release position), then mean = sum / count
monthly = monthly.isel(obs=slice(1, None)).sum("obs")

mean_maps = xr.Dataset({
    v: monthly[f"{v}_sum"] / monthly[f"{v}_count"].where(monthly[f"{v}_count"] > 0)
    for v in variables
})
mean_maps = mean_maps.compute()
mean_maps.to_netcdf("_mean_z_temp_sal_month.nc")   # optional, like the other saves
mean_maps

## Plots — one 3×4 monthly grid per variable

In [ ]:
cmaps = {"z": "cm.cm.topo", "temp": "inferno", "sal": "viridis_r"}
labels = {"z": "mean depth [m]", "temp": "mean temperature [°C]", "sal": "mean salinity"}

def plot_monthly(da, cmap, label):
    proj = ccrs.PlateCarree()
    fig, ax = plt.subplots(4, 3, figsize=(20, 8), subplot_kw=dict(projection=proj))
    ax = ax.ravel()
    vmin = float(da.min()); vmax = float(da.max())
    for i in range(12):
        dd = da.sel(start_month=i + 1)
        pcm = dd.plot.contourf(
            x="lon", y="lat", ax=ax[i], transform=proj,
            cmap=cmap, vmin=vmin, vmax=vmax, add_colorbar=False,
        )
        ax[i].coastlines()
        ax[i].add_feature(cf.LAND, facecolor="#8e9497ff", zorder=0)
        ax[i].set_title(i + 1, fontsize=10)
        ax[i].set_xticks(np.arange(np.floor(dd.lon.min()), np.ceil(dd.lon.max()) + 1e-6, 10), crs=proj)
        ax[i].set_yticks(np.arange(np.floor(dd.lat.min()), np.ceil(dd.lat.max()) + 1e-6, 5), crs=proj)
        ax[i].xaxis.set_major_formatter(LongitudeFormatter(number_format=".0f", degree_symbol="°"))
        ax[i].yaxis.set_major_formatter(LatitudeFormatter(number_format=".0f", degree_symbol="°"))
        ax[i].tick_params(labelsize=8)
        ax[i].set_xlabel(""); ax[i].set_ylabel("")
    fig.colorbar(pcm, ax=ax, shrink=0.6, label=label)
    fig.subplots_adjust(hspace=0.3)
    fig.suptitle(label, y=0.92)
    plt.show()

In [ ]:
plot_monthly(mean_maps["z"], cmaps["z"], labels["z"])

In [ ]:
plot_monthly(mean_maps["temp"], cmaps["temp"], labels["temp"])

In [ ]:
plot_monthly(mean_maps["sal"], cmaps["sal"], labels["sal"])